# ⚡ FlashAttention

Exact attention without ever materializing the giant N×N score matrix.

> ▶︎ In Colab: **Runtime → Run all** — this notebook runs top to bottom with no setup.

In [ ]:
%pip install -q numpy

In [ ]:
import numpy as np

# Naive attention builds the full N x N matrix in (slow) memory.
def naive_attention(Q, K, V):
    scores = Q @ K.T                      # N x N — the memory hog
    w = np.exp(scores - scores.max(axis=-1, keepdims=True))
    w /= w.sum(axis=-1, keepdims=True)
    return w @ V, scores.nbytes

N, d = 512, 64
Q = K = V = np.random.randn(N, d)
_, nbytes = naive_attention(Q, K, V)
print(f'Naive stores an {N}x{N} score matrix: {nbytes / 1e6:.1f} MB')

## Try it

That matrix grows as N². FlashAttention tiles the work so it never lives in slow memory — same math, far less IO. Watch the N² blow-up.

In [ ]:
for N in [512, 2048, 8192, 32768]:
    mb = (N * N * 4) / 1e6   # float32 score matrix
    print(f'N={N:>6}: full score matrix = {mb:>9,.0f} MB')
print('FlashAttention never writes this out — the bottleneck was memory, not math.')

## Takeaway

- The attention algorithm was never the bottleneck — moving data was.
- IO-aware tiling is why long context became affordable.

## 🚀 Your move

Provoke: FlashAttention changed nothing about the math and everything about the cost. Write why 'the algorithm was never the bottleneck — the memory was' is a lesson that generalizes far beyond attention.